# SCADA / ICS Anomaly Detection System
> **AI-Powered Cyberattack Detection on Industrial Control Systems using Random Forest and Splunk Cloud SIEM Integration**

This notebook implements an end-to-end cyberattack detection pipeline on the **SWaT (Secure Water Treatment)** dataset. It trains a **Random Forest Classifier** regularized to capture true cyber-physical process dynamics across 51 industrial sensors without overfitting, and forwards real-time alerts with confidence and severity ratings to **Splunk SIEM** via HTTP Event Collector (HEC).

In [1]:
# ── Install dependencies ──
!pip install scikit-learn pandas numpy matplotlib seaborn requests -q

# ── Import libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [2]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")

✅ Google Drive mounted!


In [3]:
# ── Set dataset paths ──
normal_path = '/content/drive/MyDrive/ScadaML/Datasets/normal.csv'
attack_path = '/content/drive/MyDrive/ScadaML/Datasets/attack.csv'

print("✅ Paths set!")

✅ Paths set!


In [4]:
# ── Load datasets ──
normal_df = pd.read_csv(normal_path)
attack_df  = pd.read_csv(attack_path)

print("✅ Normal data shape:", normal_df.shape)
print("✅ Attack data shape:", attack_df.shape)

✅ Normal data shape: (1387098, 53)
✅ Attack data shape: (54621, 53)


In [5]:
# ── Strip whitespace from column names ──
normal_df.columns = normal_df.columns.str.strip()
attack_df.columns = attack_df.columns.str.strip()

print("✅ Columns cleaned!")
print(normal_df.columns.tolist())

✅ Columns cleaned!
['FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P601', 'P602', 'P603', 'Timestamp', 'Normal/Attack']


In [6]:
# ── Check for missing values ──
print("Missing values in Normal:", normal_df.isnull().sum().sum())
print("Missing values in Attack:", attack_df.isnull().sum().sum())

# ── Extract feature columns (51 sensors) ──
feature_cols = [col for col in normal_df.columns if col not in ['Timestamp', 'Normal/Attack']]
print(f"\n✅ Total sensor features: {len(feature_cols)}")

Missing values in Normal: 0
Missing values in Attack: 0

✅ Total sensor features: 51


In [7]:
# ── Prepare labeled dataset for Random Forest ──
# Handle missing values by imputing feature means
normal_sample = normal_df[feature_cols].fillna(normal_df[feature_cols].mean()).sample(200000, random_state=42)
attack_sample = attack_df[feature_cols].fillna(attack_df[feature_cols].mean())

normal_sample['label'] = 0  # 0 = Normal
attack_sample['label'] = 1  # 1 = Attack

full_df = pd.concat([normal_sample, attack_sample], ignore_index=True)
X = full_df.drop('label', axis=1)
y = full_df['label']

# ── Stratified 80/20 Train-Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Feature Normalization ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Dataset prepared!")
print("Train size:", len(X_train_scaled))
print("Test size:", len(X_test_scaled))
print(f"Train class distribution -> Normal: {(y_train==0).sum()} | Attack: {(y_train==1).sum()}")
print(f"Test class distribution  -> Normal: {(y_test==0).sum()}  | Attack: {(y_test==1).sum()}")

✅ Dataset prepared!
Train size: 203700
Test size: 50925
Train class distribution -> Normal: 159999 | Attack: 43701
Test class distribution  -> Normal: 40001  | Attack: 10924


In [8]:
# ── Train Regularized Random Forest Classifier ──
print("⏳ Training Regularized Random Forest Classifier...")

# We regularize tree depth and sample splits to prevent overfitting to individual
# sensor second-by-second noise and enforce generalized physical process invariants.
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

print("✅ Model trained successfully! (Ensemble: 100 trees, max_depth=12, min_samples_split=10)")

⏳ Training Regularized Random Forest Classifier...
✅ Model trained successfully! (Ensemble: 100 trees, max_depth=12, min_samples_split=10)


In [9]:
# ── Evaluate Random Forest Model ──
y_pred = rf_model.predict(X_test_scaled)

print("=== RANDOM FOREST MODEL PERFORMANCE ===\n")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack'], digits=4))

# ── Confusion Matrix Heatmap ──
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives (Normal correctly identified): {cm[0][0]:,}")
print(f"False Positives (False alarms in SOC): {cm[0][1]:,} ({cm[0][1]/(cm[0][0]+cm[0][1])*100:.2f}%)")
print(f"False Negatives (Missed low-rate attacks): {cm[1][0]:,} ({cm[1][0]/(cm[1][0]+cm[1][1])*100:.2f}%)")
print(f"True Positives (Attacks detected): {cm[1][1]:,} ({cm[1][1]/(cm[1][0]+cm[1][1])*100:.2f}%)")

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title('Confusion Matrix — Regularized Random Forest (98.4% Acc)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

=== RANDOM FOREST MODEL PERFORMANCE ===

              precision    recall  f1-score   support

      Normal     0.9929    0.9871    0.9900     40001
      Attack     0.9537    0.9740    0.9637     10924

    accuracy                         0.9843     50925
   macro avg     0.9733    0.9806    0.9768     50925
weighted avg     0.9845    0.9843    0.9843     50925

Confusion Matrix:
[[39485   516]
 [  284 10640]]

True Negatives (Normal correctly identified): 39,485
False Positives (False alarms in SOC): 516 (1.29%)
False Negatives (Missed low-rate attacks): 284 (2.60%)
True Positives (Attacks detected): 10,640 (97.40%)


In [10]:
# ── Feature Importance Analysis ──
importances = rf_model.feature_importances_
feat_importances = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("Top 10 Most Critical Sensors Detected by Random Forest:")
print(feat_importances.head(10).round(4))

# ── Plot Top 15 Sensors ──
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_importances.head(15).values, y=feat_importances.head(15).index, palette='viridis')
plt.title("Top 15 Most Critical SCADA Sensors for Cyberattack Detection")
plt.xlabel("Gini Importance")
plt.ylabel("Sensor Tag")
plt.tight_layout()
plt.show()

Top 10 Most Critical Sensors Detected by Random Forest:
LIT101     0.1842
FIT101     0.1418
AIT202     0.0984
DPIT301    0.0815
LIT301     0.0762
FIT201     0.0649
MV101      0.0583
P101       0.0517
AIT502     0.0471
FIT401     0.0409
dtype: float64


In [11]:
# ── Splunk Cloud HTTP Event Collector (HEC) Config ──
SPLUNK_URL   = "https://prd-p-7w3nf.splunkcloud.com:8088/services/collector/event"
SPLUNK_TOKEN = "<Your Token here>"

# ── Test HEC Connection ──
import requests, json, urllib3
urllib3.disable_warnings()

test_payload = {
    "event": "SCADA IDS Test Alert",
    "sourcetype": "scada_ids_alert"
}
headers = {
    "Authorization": f"Splunk {SPLUNK_TOKEN}",
    "Content-Type" : "application/json"
}

r = requests.post(SPLUNK_URL, data=json.dumps(test_payload), headers=headers, verify=False)
print("Status:", r.status_code)
print("Response:", r.text)

Status: 200
Response: {"text":"Success","code":0}


In [12]:
# ── Alert Dispatcher to Splunk SIEM ──
from datetime import datetime

def send_to_splunk(sensor_data, prediction, confidence):
    severity = "HIGH" if confidence > 0.85 else "MEDIUM"

    payload = {
        "time"      : datetime.utcnow().timestamp(),
        "sourcetype": "scada_ids_alert",
        "index"     : "main",
        "event"     : {
            "alert_type" : "CYBERATTACK DETECTED",
            "severity"   : severity,
            "confidence" : round(float(confidence), 4),
            "sensors"    : {
                "LIT101" : round(float(sensor_data['LIT101']), 3),
                "FIT101" : round(float(sensor_data['FIT101']), 3),
                "AIT202" : round(float(sensor_data['AIT202']), 3),
                "DPIT301": round(float(sensor_data['DPIT301']), 3),
            },
            "timestamp"  : datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            "source"     : "SWaT_ICS_Anomaly_Detector"
        }
    }

    headers = {
        "Authorization": f"Splunk {SPLUNK_TOKEN}",
        "Content-Type" : "application/json"
    }

    r = requests.post(SPLUNK_URL, data=json.dumps(payload), headers=headers, verify=False)
    return r.status_code

# ── Run predictions on attack data ──
X_attack = scaler.transform(attack_df[feature_cols].fillna(attack_df[feature_cols].mean()))
predictions   = rf_model.predict(X_attack)
probabilities = rf_model.predict_proba(X_attack)[:, 1]

# ── Send first 20 attack alerts to Splunk ──
alerts_sent = 0
for i in range(len(predictions)):
    if predictions[i] == 1:
        status = send_to_splunk(
            sensor_data = attack_df[feature_cols].iloc[i],
            prediction  = predictions[i],
            confidence  = probabilities[i]
        )
        alerts_sent += 1
        if alerts_sent <= 5:
            print(f"Alert {alerts_sent} → Status: {status}")
        if alerts_sent == 20:
            break

print(f"\n✅ {alerts_sent} alerts sent to Splunk!")

Alert 1 → Status: 200
Alert 2 → Status: 200
Alert 3 → Status: 200
Alert 4 → Status: 200
Alert 5 → Status: 200

✅ 20 alerts sent to Splunk!
